In [1]:
import pandas as pd
import numpy as np

# 1. Reading files
file_path = 'input/Normalized_Data_only_origin.csv'
df = pd.read_csv(file_path)

# 2. Calculate the maximum value
max_id = df['id'].max()
max_row = df['row_index'].max()
max_col = df['col_index'].max()

# 3. Print results
print(f"File read successful!")
print(f"ID Max: {max_id}")
print(f"Row Index Max: {max_row}")
print(f"Col Index Max: {max_col}")

# preview
print("\data preview：")
print(df.head())

File read successful!
ID Max: 267999
Row Index Max: 470
Col Index Max: 568
\data preview：
   id  row_index  col_index  transport_None  green15  15mins  employe  \
0   1          0          0             NaN      NaN     NaN      NaN   
1   2          1          0             NaN      NaN     NaN      NaN   
2   3          2          0             NaN      NaN     NaN      NaN   
3   4          3          0             NaN      NaN     NaN      NaN   
4   5          4          0             NaN      NaN     NaN      NaN   

   gps_join_osm_id_count  landmark_osm_id_count  \
0                    NaN                    NaN   
1                    NaN                    NaN   
2                    NaN                    NaN   
3                    NaN                    NaN   
4                    NaN                    NaN   

   integration_intergation_T1024 Integration R800 metric_mean  ...  Christian  \
0                                                NaN           ...        NaN   
1 

In [2]:
def build_lookup_dictionary(dataframe):

    # 1. Determine which are the data columns
    # Based on the file structure: column 0 is id, column 1 is row, and column 2 is col.
    # Therefore, the data starts from column 3 (index 3).
    data_columns = dataframe.columns[3:]
    print(f"Detected data indicator columns (Number:{len(data_columns)}): {data_columns.tolist()}")

    # 2. Create a dictionary
    # Use zip to combine row_index and col_index into a key.
    # Convert the values of the data column into a list as the value.
    # This method is much faster than the iterrows loop.
    keys = zip(dataframe['row_index'], dataframe['col_index'])
    values = dataframe[data_columns].values.tolist()

    lookup_dict = dict(zip(keys, values))
    
    return lookup_dict, data_columns

# Execute dictionary creation
data_dict, data_col_names = build_lookup_dictionary(df)

# Define lookup function
def get_data_value(x, y, i):
    """
    Input (x, y, i), return the corresponding data
    x: row_index
    y: col_index
    i: the i-th data column (0-indexed, corresponds to A, B, C...)
    """
    try:
        # Get the list of all data at these coordinates
        row_values = data_dict[(x, y)]
        
        # Check if i is out of range for the data columns
        if i < 0 or i >= len(row_values):
            return f"Error: Index i={i} out of range (maximum index is {len(row_values)-1})"
            
        # Return the specific value
        val = row_values[i]
        
        # Optional: Handle null values (NaN), if you need to return None or 0 you can modify here
        if val is None or (isinstance(val, float) and np.isnan(val)):
            return "NaN (Null value)"
            
        return val
        
    except KeyError:
        return f"Error: Data not found for coordinates row_index={x}, col_index={y}"

Detected data indicator columns (Number:31): ['transport_None', 'green15', '15mins', 'employe', 'gps_join_osm_id_count', 'landmark_osm_id_count', 'integration_intergation_T1024 Integration R800 metric_mean', 'choice_15m_Metric Choice R800 metric_mean', 'BuildingHeight_(mean)', 'BuildingVolumn', 'BuildingArea', 'BuildingDensity_(area)', 'BuildingAge_(mean)', 'VacantHouseCount', 'vnl_NightLightmean', 'income', 'employment', 'activity limited%', 'work population', 'BAME%_re', 'no-native English speaker %', 'Christian', 'Hindu', 'Muslim', 'Acommodation_owned', 'Acommodation_Private rented', 'Acommodation_social rented', 'Educational_facilities_name_count', 'Healthcare_facilities_name_count', 'supermarket_name_count', 'UGS_Green space per capita_None']


In [3]:
df.columns[3:]

Index(['transport_None', 'green15', '15mins', 'employe',
       'gps_join_osm_id_count', 'landmark_osm_id_count',
       'integration_intergation_T1024 Integration R800 metric_mean',
       'choice_15m_Metric Choice R800 metric_mean', 'BuildingHeight_(mean)',
       'BuildingVolumn', 'BuildingArea', 'BuildingDensity_(area)',
       'BuildingAge_(mean)', 'VacantHouseCount', 'vnl_NightLightmean',
       'income', 'employment', 'activity limited%', 'work population',
       'BAME%_re', 'no-native English speaker %', 'Christian', 'Hindu',
       'Muslim', 'Acommodation_owned', 'Acommodation_Private rented',
       'Acommodation_social rented', 'Educational_facilities_name_count',
       'Healthcare_facilities_name_count', 'supermarket_name_count',
       'UGS_Green space per capita_None'],
      dtype='object')

In [4]:
'''
# --- 测试代码 ---
print("\n--- 测试查询 ---")
# 这里的测试用例基于您的文件前几行
# 例如：测试 row_index=0, col_index=0, 第0个数据指标
test_x, test_y, test_i = 0, 0, 0
result = get_data_value(test_x, test_y, test_i)
print(f"查询 (x={test_x}, y={test_y}, i={test_i}) -> 对应列名: {data_col_names[test_i]}")
print(f"结果: {result}")
'''


# Test a point with actual data (based on the file preview, data may appear later).
test_x2, test_y2 = 20, 0
result2 = get_data_value(test_x2, test_y2, 2)
print(f"\n check (x={test_x2}, y={test_y2}, i=2)")
print(f"result: {result2}")


 check (x=20, y=0, i=2)
result: NaN (Null value)


In [5]:
A=[]
for x in range(1,max_row):
    for y in range(1, max_col):
        if isinstance(get_data_value(x, y, 2), (int, float, complex)):
            a1= isinstance(get_data_value(x+1, y, 2), (int, float, complex))
            a2= isinstance(get_data_value(x-1, y, 2), (int, float, complex))
            a3= isinstance(get_data_value(x, y+1, 2), (int, float, complex))
            a4= isinstance(get_data_value(x, y-1, 2), (int, float, complex))
            if a1+a2+a3+a4 ==4:
                Res=((get_data_value(x+1, y, 2)+get_data_value(x-1, y, 2)+get_data_value(x, y+1, 2)+get_data_value(x, y-1, 2))/4) -get_data_value(x, y, 2)
            if a1+a2+a3+a4 ==3:
                R1=0
                if a1:
                    R1+=get_data_value(x+1, y, 2)
                if a2:
                    R1+=get_data_value(x-1, y, 2)
                if a3:
                    R1+=get_data_value(x, y+1, 2)
                if a4:
                    R1+=get_data_value(x, y-1, 2)
                Res=R1/3 -get_data_value(x, y, 2)
            A.append([x, y, Res])

In [6]:
A=[]
for i in range(len(df.columns[3:])):
    A.append([])
    for x in range(1,max_row):
        for y in range(1, max_col):
            if isinstance(get_data_value(x, y, i), (int, float, complex)):
                a1= isinstance(get_data_value(x+1, y, i), (int, float, complex))
                a2= isinstance(get_data_value(x-1, y, i), (int, float, complex))
                a3= isinstance(get_data_value(x, y+1, i), (int, float, complex))
                a4= isinstance(get_data_value(x, y-1, i), (int, float, complex))
                if a1+a2+a3+a4 ==4:
                    Res=((get_data_value(x+1, y, i)+get_data_value(x-1, y, i)+get_data_value(x, y+1, i)+get_data_value(x, y-1, i))/4) -get_data_value(x, y, i)
                if a1+a2+a3+a4 ==3:
                    R1=0
                    if a1:
                        R1+=get_data_value(x+1, y, i)
                    if a2:
                        R1+=get_data_value(x-1, y, i)
                    if a3:
                        R1+=get_data_value(x, y+1, i)
                    if a4:
                        R1+=get_data_value(x, y-1, i)
                    Res=R1/3 -get_data_value(x, y, i)
                A[i].append([x, y, Res])
    df_lap = pd.DataFrame(A[i], columns=['row_index', 'col_index', "Laplus "+df.columns[3:][i]])

    # 4. Merge the new data back into the original DataFrame
    
    df = pd.merge(df, df_lap, on=['row_index', 'col_index'], how='left')
    
    # 5. check output
    print("sucess")
    # View the row where data was just entered (e.g., row=1, col=17).
    check_row = df[(df['row_index'] == 1) & (df['col_index'] == 17)]
    print("\n input (row=1, col=17):")
    print(check_row[['row_index', 'col_index', "Laplus "+df.columns[3:][i]]])

sucess

 input (row=1, col=17):
      row_index  col_index  Laplus transport_None
8008          1         17                    NaN
sucess

 input (row=1, col=17):
      row_index  col_index  Laplus green15
8008          1         17             NaN
sucess

 input (row=1, col=17):
      row_index  col_index  Laplus 15mins
8008          1         17            NaN
sucess

 input (row=1, col=17):
      row_index  col_index  Laplus employe
8008          1         17             NaN
sucess

 input (row=1, col=17):
      row_index  col_index  Laplus gps_join_osm_id_count
8008          1         17                           NaN
sucess

 input (row=1, col=17):
      row_index  col_index  Laplus landmark_osm_id_count
8008          1         17                           NaN
sucess

 input (row=1, col=17):
      row_index  col_index  \
8008          1         17   

      Laplus integration_intergation_T1024 Integration R800 metric_mean  
8008                                                NaN  

In [7]:
df.to_csv('Laplus 333.csv', index=False)
df.columns[3:]

Index(['transport_None', 'green15', '15mins', 'employe',
       'gps_join_osm_id_count', 'landmark_osm_id_count',
       'integration_intergation_T1024 Integration R800 metric_mean',
       'choice_15m_Metric Choice R800 metric_mean', 'BuildingHeight_(mean)',
       'BuildingVolumn', 'BuildingArea', 'BuildingDensity_(area)',
       'BuildingAge_(mean)', 'VacantHouseCount', 'vnl_NightLightmean',
       'income', 'employment', 'activity limited%', 'work population',
       'BAME%_re', 'no-native English speaker %', 'Christian', 'Hindu',
       'Muslim', 'Acommodation_owned', 'Acommodation_Private rented',
       'Acommodation_social rented', 'Educational_facilities_name_count',
       'Healthcare_facilities_name_count', 'supermarket_name_count',
       'UGS_Green space per capita_None', 'Laplus transport_None',
       'Laplus green15', 'Laplus 15mins', 'Laplus employe',
       'Laplus gps_join_osm_id_count', 'Laplus landmark_osm_id_count',
       'Laplus integration_intergation_T1024 Int

In [12]:
import pandas as pd

file_path1 = 'Laplus 333.csv'
df = pd.read_csv(file_path1)

# 1. Prepare data (assuming previous steps have been run, df_merged already exists)
# If running independently, please ensure df_merged already contains the LapA column
# (For code completeness, repeating the merge step here; if already merged in actual run, this can be skipped)
# df_merged = pd.merge(df, df_lap, on=['row_index', 'col_index'], how='left')

# --- Modified cleaning logic ---

# [Step 1] Set filter conditions: specify which columns "must absolutely not be null"
# You can fill in the column names in this list.
# For example: If you only care that the original data 'adj Choice' must have a value, but don't care if 'LapA' has a value
check_columns = ['income', 'employment', 'activity limited%', 'work population',
       'BAME%_re', 'no-native English speaker %', 'Christian', 'Hindu',
       'Muslim', 'Acommodation_owned', 'Acommodation_Private rented',
       'Acommodation_social rented',
       'Laplus income', 'Laplus employment',
       'Laplus activity limited%', 'Laplus work population', 'Laplus BAME%_re',
       'Laplus no-native English speaker %', 'Laplus Christian',
       'Laplus Hindu', 'Laplus Muslim', 'Laplus Acommodation_owned',
       'Laplus Acommodation_Private rented',
       'Laplus Acommodation_social rented'] 

# If you want both 'adj Choice' and 'LapA' to have values, then write:
# check_columns = ['adj Choice', 'LapA']

print(f"Filtering data based on the following columns: {check_columns}")

# [Step 2] Execute deletion
# The subset parameter specifies that only these columns are checked
df_cleaned = df.dropna(subset=check_columns)

print(f"Number of rows before filtering: {len(df)}")
print(f"Number of rows after filtering: {len(df_cleaned)}")

# [Step 3] Fill remaining null values
# Fill all remaining NaNs with 0
df_final = df_cleaned.fillna(0)

Filtering data based on the following columns: ['income', 'employment', 'activity limited%', 'work population', 'BAME%_re', 'no-native English speaker %', 'Christian', 'Hindu', 'Muslim', 'Acommodation_owned', 'Acommodation_Private rented', 'Acommodation_social rented', 'Laplus income', 'Laplus employment', 'Laplus activity limited%', 'Laplus work population', 'Laplus BAME%_re', 'Laplus no-native English speaker %', 'Laplus Christian', 'Laplus Hindu', 'Laplus Muslim', 'Laplus Acommodation_owned', 'Laplus Acommodation_Private rented', 'Laplus Acommodation_social rented']
Number of rows before filtering: 267999
Number of rows after filtering: 166993


In [9]:
df_final.to_csv('Laplus Cleaned 11.csv', index=False)

In [13]:
# --- Step 3: Data cleaning ---

# Delete rows containing any null values (NaN)
# This step will be very strict: as long as there is any empty cell in a row (including an empty LapA column, or empty original data columns), the entire row will be deleted.
# The result will only keep those rows that "have data in ListA, and also have data in the original file".
df_final = df.dropna(how='any')

# Print the results to check
print(f"Number of rows before cleaning: {len(df)}")
print(f"Number of rows after cleaning: {len(df_final)}")
print("\nData preview after cleaning:")
print(df_final.head())

Number of rows before cleaning: 267999
Number of rows after cleaning: 0

Data preview after cleaning:
Empty DataFrame
Columns: [id, row_index, col_index, transport_None, green15, 15mins, employe, gps_join_osm_id_count, landmark_osm_id_count, integration_intergation_T1024 Integration R800 metric_mean, choice_15m_Metric Choice R800 metric_mean, BuildingHeight_(mean), BuildingVolumn, BuildingArea, BuildingDensity_(area), BuildingAge_(mean), VacantHouseCount, vnl_NightLightmean, income, employment, activity limited%, work population, BAME%_re, no-native English speaker %, Christian, Hindu, Muslim, Acommodation_owned, Acommodation_Private rented, Acommodation_social rented, Educational_facilities_name_count, Healthcare_facilities_name_count, supermarket_name_count, UGS_Green space per capita_None, Laplus transport_None, Laplus green15, Laplus 15mins, Laplus employe, Laplus gps_join_osm_id_count, Laplus landmark_osm_id_count, Laplus integration_intergation_T1024 Integration R800 metric_mean,

In [11]:
df_final.to_csv('Updated_Sample_Data.csv', index=False)